In [1]:
!pip install  -U -q trl peft datasets huggingface_hub hf_transfer wandb weave

In [2]:
from huggingface_hub import login

login(token="insert token")

In [3]:
import wandb

wandb.login(key="insert token")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: occultainsights to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import torch

# Verify CUDA availability and display GPU specifications
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    # Display current GPU details for training optimization
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    # Provide guidance for enabling GPU in Colab
    print("⚠️  No GPU available. This notebook requires a GPU for efficient training.")
    print("In Colab: Runtime → Change runtime type → Hardware accelerator → GPU")

CUDA available: True
Number of GPUs: 1
Current GPU: 0
GPU name: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 102.0 GB


In [5]:
import os
import csv
import shutil
from pathlib import Path
from typing import List

import torch
import weave
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig
from datasets import load_dataset

import re

In [40]:
# =====================================================================================
# Config
# =====================================================================================

# HF model to fine-tune 
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Basic training hyperparams
RESUME_FROM_CHECKPOINT = True
# CHECKPOINT_DIR="./trl_grpo_outputs_01"  
CHECKPOINT_DIR="./trl_grpo_checkpoint_IL_blog_01"
LOG_DIR = "./logs/grpo-conll-lora"
HF_REPO_ID = 'Priyanlc'
OUTPUT_DIR = "./trl_grpo_output_IL_blog_01"
         
NUM_EPOCHS = 1.0
#PER_DEVICE_BATCH_SIZE = 8 change for 24 G VRAM
PER_DEVICE_BATCH_SIZE = 32  # for A100 80 GB
#PER_DEVICE_EVAL_BATCH_SIZE = 8 change for 24 G VRAM
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 4 #reduce to 2 for a 24 G VRAM GPU
LEARNING_RATE = 5e-6
TEMPERATURE = 0.6

MAX_PROMPT_LENGTH = 512
MAX_COMPLETION_LENGTH = 512
NUM_GENERATIONS = 8      # groups per prompt for GRPO
BETA = 0.01

RUN_NAME = f"{MODEL_NAME}-conll-grpo-cuda"

RANK = 32
ALPHA = 64

# =====================================================================
# Eval dataset: 
# =====================================================================
EVAL_SAMPLES = 40
PRINT_FIRST_N_EVAL = 4
MAX_TRAINING_STEPS = 300

## Prompt

In [41]:
reasoning_start = "<reasoning>"
reasoning_end = "</reasoning>"
solution_start = "<answer>"
solution_end = "</answer>"

system_prompt = f"""
You are an information extraction system.

Your task is to assign a BIO Named Entity tag to each input token.

Entity types:
- PER (Person)
- ORG (Organization)
- LOC (Location)
- MISC (Miscellaneous)

BIO rules:
- Each entity must start with B-<TYPE>
- Tokens inside the same entity must use I-<TYPE>
- Tokens outside any entity must use O
- Output exactly one label per input token
- Do not add, remove, or reorder tokens

Always answer in this exact format:

{reasoning_start}
Briefly explain the main entity decisions you made.
Do NOT explain every token.
{reasoning_end}

{solution_start}
TOKEN<TAB>LABEL
(one token per line, in the same order as input)
{solution_end}

Rules:
- Put ALL reasoning only between {reasoning_start} and {reasoning_end}.
- The {solution_start} section must contain ONLY token–label pairs.
- Use a TAB character between token and label.
- Do NOT output anything after {solution_end}.

### Example

Tokens:
EU rejects German call to boycott British lamb .

Output:

{reasoning_start}
EU is an organization. German and British are nationality adjectives and are labeled as MISC. Other tokens are not named entities.
{reasoning_end}

{solution_start}
EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O
{solution_end}
### Task

Tokens:
<INSERT TOKENS HERE>

Output:
"""

print("✅ Format tokens and system prompt defined")
print(f"   Reasoning format: {reasoning_start} ... {reasoning_end}")
print(f"   Solution format: {solution_start} ... {solution_end}")

✅ Format tokens and system prompt defined
   Reasoning format: <reasoning> ... </reasoning>
   Solution format: <answer> ... </answer>


### Prepares the CoNLL-2003 NER dataset for training in 2 stages 
Stage1:  Parses raw CoNLL-formatted files into sentence-level token and NER tag pairs, then converts them into structured records suitable for dataset construction and downstream training.

In [42]:
from datasets import Dataset, Features, Sequence, Value

def read_conll(path):
    sentences = []
    sentence = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue

            token, pos, chunk, ner = line.split()
            sentence.append((token, ner))

        if sentence:
            sentences.append(sentence)

    return sentences

def conll_to_records(sentences):
    records = []

    for sentence in sentences:
        records.append({
            "tokens": [tok for tok, _ in sentence],
            "ner_tags": [ner for _, ner in sentence],
        })

    return records


features = Features({
    "tokens": Sequence(Value("string")),
    "ner_tags": Sequence(Value("string")),
})

train_sentences = read_conll("conll_data/eng.train")
val_sentences   = read_conll("conll_data/eng.testa")
test_sentences  = read_conll("conll_data/eng.testb")

train_ds = Dataset.from_list(
    conll_to_records(train_sentences),
    features=features,
)

val_ds = Dataset.from_list(
    conll_to_records(val_sentences),
    features=features,
)

test_ds = Dataset.from_list(
    conll_to_records(test_sentences),
    features=features,
)

Stage2: Defining an explicit token–label schema, loading train/validation/test splits, and converting raw CoNLL sentences into Hugging Face `Dataset` objects. Each split is then preprocessed into the exact format required by the GRPO training loop, ensuring strict token–label alignment and consistent structure across datasets.

In [43]:
def render_tokens(tokens):
    return " ".join(tokens)

def render_gold_answer(tokens, ner_tags):
    lines = []
    for tok, tag in zip(tokens, ner_tags):
        lines.append(f"{tok}\t{tag}")
    return "\n".join(lines)

def process_conll_example(example):
    """
    Convert CoNLL NER example to conversation format for GRPO training. 
    """
    tokens = example["tokens"]
    ner_tags = example["ner_tags"]

    # User input (tokens only)
    user_input = f"""Tokens:
{render_tokens(tokens)}"""

    # Ground-truth answer (BIO labels only)
    gold_answer = render_gold_answer(tokens, ner_tags)

    # Conversation structure (mirrors GSM8K)
    prompt = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input},
    ]

    return {
        "prompt": prompt,        # Input conversation
        "answer": gold_answer,   # Ground truth BIO labels
    }

processed_train = train_ds.map(
    process_conll_example,
    remove_columns=train_ds.column_names,
)

processed_test = test_ds.map(
    process_conll_example,
    remove_columns=test_ds.column_names,
)

processed_val = val_ds.map(
    process_conll_example,
    remove_columns=val_ds.column_names,
)

Map:   0%|          | 0/14987 [00:00<?, ? examples/s]

Map:   0%|          | 0/3684 [00:00<?, ? examples/s]

Map:   0%|          | 0/3466 [00:00<?, ? examples/s]

In [44]:
print(f"✅ Dataset loaded and processed!")
print(f"📊 Training examples: {len(processed_train):,}")
print(f"🎯 Sample question: {processed_train[0]['prompt'][1]['content']}...")
print(f"🎯 Sample answer: {processed_train[0]['answer']}")

# Show structure of first example for verification
print(f"\n📋 Example structure:")
print(f"   Prompt: {len(processed_train[0]['prompt'])} messages (system + user)")
print(f"   Answer: {processed_train[0]['answer']} (ground truth for rewards)")
  
eval_dataset = processed_test.select(range(min(EVAL_SAMPLES, len(processed_test))))

print(f"✅ Eval dataset prepared with {len(eval_dataset)} examples")
print(f"🎯 Sample eval question: {eval_dataset[1]['prompt'][1]['content']}")
print(f"🎯 Sample eval answer: {eval_dataset[0]['answer']}")

✅ Dataset loaded and processed!
📊 Training examples: 14,987
🎯 Sample question: Tokens:
-DOCSTART-...
🎯 Sample answer: -DOCSTART-	O

📋 Example structure:
   Prompt: 2 messages (system + user)
   Answer: -DOCSTART-	O (ground truth for rewards)
✅ Eval dataset prepared with 40 examples
🎯 Sample eval question: Tokens:
SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
🎯 Sample eval answer: -DOCSTART-	O


Imports and configures the core libraries required for GRPO-based training, including model loading, tokenization, parameter-efficient fine-tuning, dataset handling, reward evaluation, and logging.


In [45]:
import re       # Regex patterns for reward functions

# GRPO training components
from trl import GRPOConfig, GRPOTrainer

# Model and tokenization
from transformers import (
    AutoModelForCausalLM,   # Causal language model loading
    AutoTokenizer,          # Text tokenization
    BitsAndBytesConfig,     # Quantization configuration
)

# Parameter-efficient fine-tuning
from peft import LoraConfig, get_peft_model, TaskType

# Dataset handling
from datasets import load_dataset

# Logging configuration
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [46]:
# Select model optimized for instruction-following and reasoning
model_name = MODEL_NAME                   # 3B parameter model balances capability and memory usage
max_seq_length = 2048                     # Token limit (reduce if OOM)

print(f"Loading model: {model_name}")
print(f"Max sequence length: {max_seq_length}")

# Load model with automatic device mapping
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",                   # Auto-distribute across available GPUs/CPU
    trust_remote_code=True,              # Allow custom model code execution
    dtype=torch.bfloat16,
)

# Load corresponding tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True               # Allow custom tokenizer code
)

# Ensure tokenizer has proper padding token for batch processing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading model: Qwen/Qwen2.5-3B-Instruct
Max sequence length: 2048


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [47]:
# Configure LoRA 
lora_config = LoraConfig(
    r=RANK,  
    lora_alpha=ALPHA,  
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("🔧 Applying LoRA adaptation to model...")
model = get_peft_model(model, lora_config)

print(f"✅ Model loaded successfully!")
print(f"📊 Model parameters: ~{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# # Display parameter efficiency
# print("📊 LoRA Training Parameters Summary:")
model.print_trainable_parameters()  # Shows trainable vs total parameters

🔧 Applying LoRA adaptation to model...
✅ Model loaded successfully!
📊 Model parameters: ~3145.8M
trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


In [48]:
def count_trainable_parameters(model):
    """
    Another generic function to count the trainable parameters. 
    """
    trainable = 0
    total = 0
    for _, param in model.named_parameters():
        num = param.numel()
        total += num
        if param.requires_grad:
            trainable += num
    return trainable, total


trainable, total = count_trainable_parameters(model)
print(f"Trainable params: {trainable:,}")
print(f"Total params:     {total:,}")
print(f"Trainable %:      {100 * trainable / total:.4f}%")

Trainable params: 59,867,136
Total params:     3,145,805,824
Trainable %:      1.9031%


In [49]:
def apply_template(example):
    """
    Applies the model-specific chat template to the prompt, converting it into
    the formatted input expected by the causal language model during GRPO training.
    This is the model’s required input format.
    """
    
    example["prompt"] = tokenizer.apply_chat_template(
        example["prompt"], tokenize=False, add_generation_prompt=True
    )
    return example

processed_train = processed_train.map(apply_template)
processed_train

Map:   0%|          | 0/14987 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'answer'],
    num_rows: 14987
})

## Reward functions 
### Utilities (Shared Helpers)

In [50]:
import re

def extract_section(text, start_tag, end_tag):
    pattern = re.compile(
        re.escape(start_tag) + r"(.*?)" + re.escape(end_tag),
        re.DOTALL,
    )
    match = pattern.search(text)
    return match.group(1).strip() if match else None

In [51]:
def parse_answer(answer_text):
    """
    Returns: List[(token, label)]
    """
    pairs = []
    for line in answer_text.splitlines():
        if not line.strip():
            continue
        try:
            tok, label = line.split("\t")
            pairs.append((tok, label))
        except ValueError:
            return None
    return pairs

In [52]:
def extract_completion_text(completion):
    """
    Handles GRPO chat-style completions safely.
    """
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        print("\n" + "-" * 80)
        print("extract_completion_text => ")
        print(completion)
        print("-" * 80)
        for msg in reversed(completion):
            if msg.get("role") == "assistant":
                return msg.get("content", "")

    return ""

## Reward: Output Format (Hard Constraint)
### Goal: Ensure the model respects the contract.

In [53]:
def reward_format(completions, **kwargs):
    # In TRL 2026, completions is a list of generated sequences
    rewards = []

    for completion in completions:
        # 1. Safely extract the string content
        text = extract_completion_text(completion)
        
        # 2. Handle cases where extraction fails (returns empty string)
        if not text or not isinstance(text, str):
            rewards.append(-1.0)
            continue

        # 3. Extract your custom sections
        reasoning = extract_section(text, "<reasoning>", "</reasoning>")
        answer = extract_section(text, "<answer>", "</answer>")

        # 4. Strict validation
        if reasoning and answer:
            rewards.append(1.0)
        else:
            # Penalty for missing scaffolding
            rewards.append(-1.0)

    return rewards

## Reward: Token Count & Alignment (Hard)
### Goal: Detect missing, extra, or reordered tokens.

In [54]:
def reward_token_alignment(completions, **kwargs):
    gold_answers = kwargs["answer"]
    rewards = []

    for i, (completion, gold) in enumerate(zip(completions, gold_answers)):
        # Extract assistant text from chat-style completion
        text = extract_completion_text(completion)
        answer_text = extract_section(text, "<answer>", "</answer>")
        
        if answer_text is None:
            rewards.append(0.0)
            continue

        pred = parse_answer(answer_text)
        
        if pred is None:
            rewards.append(0.0)
            continue

        gold_lines = gold.splitlines()
        if len(gold_lines) == 0:
            rewards.append(0.0)
            continue

        # 🔑 BOOTSTRAP ALIGNMENT REWARD
        length_ratio = min(len(pred), len(gold_lines)) / max(len(pred), len(gold_lines))
        rewards.append(length_ratio)

    return rewards

## Reward: BIO Validity (Hard)
### Enforce structural correctness of BIO tagging.

In [55]:
def is_valid_bio(labels):
    prev = "O"
    for label in labels:
        if label == "O":
            prev = label
            continue

        if "-" not in label:
            return False

        prefix, ent = label.split("-", 1)

        if prefix == "B":
            prev = label
        elif prefix == "I":
            if prev == "O" or prev.split("-", 1)[1] != ent:
                return False
            prev = label
        else:
            return False

    return True

In [56]:
def reward_bio_validity(completions, **kwargs):
    rewards = []

    for completion in completions:
        # --- FIX: extract assistant text from chat-style completion ---
        text = extract_completion_text(completion)

        answer_text = extract_section(text, "<answer>", "</answer>")
        if answer_text is None:
            rewards.append(-1.0)
            continue

        pred = parse_answer(answer_text)
        if pred is None:
            rewards.append(-1.0)
            continue

        labels = [label for _, label in pred]

        rewards.append(1.0 if is_valid_bio(labels) else -1.0)

    return rewards

## Reward: Entity-Level Correctness (Soft, Core Learning Signal)
### Goal: Move from “valid BIO” → “correct entities”.
#### Simple token-level accuracy (safe starting point):

In [57]:
def reward_token_accuracy(completions, **kwargs):
    gold_answers = kwargs["answer"]
    rewards = []

    for completion, gold in zip(completions, gold_answers):
        # --- FIX: extract assistant text from chat-style completion ---
        text = extract_completion_text(completion)

        answer_text = extract_section(text, "<answer>", "</answer>")
        if answer_text is None:
            rewards.append(0.0)
            continue

        pred = parse_answer(answer_text)
        if pred is None:
            rewards.append(0.0)
            continue

        gold_pairs = parse_answer(gold)
        if gold_pairs is None or len(gold_pairs) == 0:
            rewards.append(0.0)
            continue

        correct = sum(
            1 for (_, pred_label), (_, gold_label)
            in zip(pred, gold_pairs)
            if pred_label == gold_label
        )

        rewards.append(correct / len(gold_pairs))

    return rewards

## Reward: Reasoning Presence (Soft)

### Goal: Encourage per-token reasoning without judging correctness (yet).

In [58]:
def reward_reasoning_presence(completions, **kwargs):
    rewards = []

    for completion in completions:
        # --- FIX: extract assistant text from chat-style completion ---
        text = extract_completion_text(completion)

        reasoning = extract_section(text, "<reasoning>", "</reasoning>")
        if reasoning and len(reasoning.strip()) > 0:
            rewards.append(0.5)
        else:
            rewards.append(0.0)

    return rewards

### Combine Rewards (Typical GRPO Setup)

In [59]:
def combined_reward_phase0(completions, **kwargs):
    r1 = reward_format(completions)
    r2 = reward_token_alignment(completions, **kwargs)
    r3 = reward_bio_validity(completions)
    r4 = reward_token_accuracy(completions, **kwargs)
    r5 = reward_reasoning_presence(completions)

    return [
        1.0*r1[i] +
        1.0*r2[i] +
        1.0*r3[i] +
        2.0*r4[i] +
        0.5*r5[i]
        for i in range(len(completions))
    ]

In [60]:
def combined_reward(completions, **kwargs):
    r_format = reward_format(completions)
    r_align  = reward_token_alignment(completions, **kwargs)
    r_bio    = reward_bio_validity(completions)
    r_acc    = reward_token_accuracy(completions, **kwargs)
    r_reason = reward_reasoning_presence(completions)

    rewards = []
    for i in range(len(completions)):
        rewards.append(
            0.2 * r_format[i] +          # already solved, keep alive
            3.0 * r_align[i] +           # PRIMARY learning signal
            3.0 * r_bio[i] +             # PRIMARY learning signal
            0.1 * r_acc[i] +             # almost ignored
            0.1 * r_reason[i]            # already solved
        )
    return rewards

In [61]:
import numpy as np
import torch
import wandb


def generate_completion_only(model, tokenizer, messages, **gen_kwargs):
    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )

    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = outputs[0][prompt_len:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


def evaluate_model_on_conll(
    model,
    tokenizer,
    eval_dataset,
    max_length=MAX_COMPLETION_LENGTH,
    temperature: float = 0.7,
    do_sample: bool = True,
    top_p: float = 0.9,
    print_first_n: int = PRINT_FIRST_N_EVAL,
    prefix: str = "eval",
    step: int | None = None,
):
    """
    Run evaluation on a CoNLL NER dataset with reasoning + BIO output
    and compute reward-based metrics.

    Args:
        model:           Causal LM (LoRA-wrapped or base).
        tokenizer:       Matching tokenizer.
        eval_dataset:    Dataset with fields: "prompt" and "answer".
        max_length:      Max new tokens to generate.
        temperature:     Sampling temperature.
        do_sample:       Whether to sample or use greedy decoding.
        top_p:           Nucleus sampling parameter.
        print_first_n:   Print input/output for first N samples only (0 = disable).
        prefix:          Prefix for metric names in W&B.
        step:            Optional global step for W&B logging.

    Returns:
        dict: Metrics dictionary.
    """
    model.eval()

    completions = []
    answers = []

    # ---------- Generate completions ----------
    for idx, example in enumerate(eval_dataset):
        prompt_messages = example["prompt"]
        gold_answer = example["answer"]
        answers.append(gold_answer)

        generated_text = generate_completion_only(
                                model,
                                tokenizer,
                                prompt_messages,
                                max_new_tokens=max_length,
                                temperature=temperature,
                                do_sample=do_sample,
                                top_p=top_p,
                                )

        if print_first_n > 0 and idx < print_first_n:
            print("\n" + "-" * 80)
            print("evaluate_model_on_conll => prompt")
            print(prompt_messages)
            print("evaluate_model_on_conll => model output")
            print("-" * 80)
            print(generated_text)
            print("=" * 80 + "\n")

        completions.append(generated_text)

    # ---------- Compute rewards / metrics ----------
    format_scores = reward_format(completions)
    
    alignment_scores = reward_token_alignment(
        completions,
        answer=answers,
    )
    
    bio_validity_scores = reward_bio_validity(completions)

    token_accuracy_scores = reward_token_accuracy(
        completions,
        answer=answers,
    )
    reasoning_scores = reward_reasoning_presence(completions)

    metrics = {
        "format_mean": float(np.mean(format_scores)),
        "format_rate": float(np.mean([s > 0 for s in format_scores])),

        "alignment_mean": float(np.mean(alignment_scores)),
        "alignment_rate": float(np.mean([s > 0 for s in alignment_scores])),

        "bio_validity_mean": float(np.mean(bio_validity_scores)),
        "bio_validity_rate": float(np.mean([s > 0 for s in bio_validity_scores])),

        "token_accuracy_mean": float(np.mean(token_accuracy_scores)),
        "token_accuracy_exact": float(
            np.mean([s == 1.0 for s in token_accuracy_scores])
        ),

        "reasoning_presence_rate": float(
            np.mean([s > 0 for s in reasoning_scores])
        ),
    }

    combined = combined_reward(
        completions,
        answer=answers,
    )
    metrics["combined_reward_mean"] = float(np.mean(combined))

    if wandb.run is not None:
        log_dict = {f"{prefix}/{k}": v for k, v in metrics.items()}
        if step is not None:
            wandb.log(log_dict, step=step)
        else:
            wandb.log(log_dict)

    return metrics

In [62]:
# Configure GRPO training parameters 
training_args = GRPOConfig(
    # Learning parameters optimized for reasoning tasks
    learning_rate= LEARNING_RATE,  # Conservative LR to prevent destabilizing reasoning
    
    # Memory-efficient batch configuration
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,   # Small batch for GPU memory constraints
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,   # Effective batch size = 2 * 8 = 16
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    bf16=True,     

    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    
    # GRPO-specific knobs
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    beta = BETA,
    
    # Training duration and monitoring
    max_steps=MAX_TRAINING_STEPS,     
    logging_steps=1,                 # Log metrics every step for close monitoring
    save_steps=25,
    
    # Stability and output configuration
    output_dir=CHECKPOINT_DIR,
    max_grad_norm=0.1,               # Aggressive gradient clipping for stable training
    report_to="wandb",               # use wandb for experiment tracking 

    remove_unused_columns=False,
    run_name=RUN_NAME
)

<string>:196: FutureWarning: The `max_prompt_length` argument is deprecated and will be removed in version 0.28.0. You should instead filter your dataset before training to ensure that prompts do not exceed your desired length.


In [63]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        reward_format,            # Keep output scaffolding stable
        reward_token_alignment,   # PRIMARY: exact token ↔ label alignment
        # reward_bio_validity,
        # reward_token_accuracy,
        reward_reasoning_presence, # Keep reasoning alive, low influenc 
        
    ],
    args=training_args,
    train_dataset=processed_train,
)

print("🚦 Phase 1 GRPO Trainer (Structure-Only) initialized")
print(f"📊 Training dataset: {len(processed_train):,} examples")
print("🎯 Active rewards: format | alignment | BIO | reasoning")
print(f"🔄 Generations per step: {training_args.num_generations}")

The model is already on multiple devices. Skipping the move to device specified in `args`.


🚦 Phase 1 GRPO Trainer (Structure-Only) initialized
📊 Training dataset: 14,987 examples
🎯 Active rewards: format | alignment | BIO | reasoning
🔄 Generations per step: 8


In [64]:
from typing import Optional
from huggingface_hub import HfApi, create_repo
from transformers import PreTrainedModel, PreTrainedTokenizerBase


def save_policy_model_to_hf(
    trainer,
    tokenizer: PreTrainedTokenizerBase,
    repo_id: str,
    output_dir: str,
    private: bool = False,
) -> Optional[PreTrainedModel]:
    """Merge LoRA policy into the base model, save locally, and push to HF Hub.

    Args:
        trainer:
            A GRPOTrainer (or compatible TRL trainer) whose `.model` is a PEFT
            model with LoRA adapters applied.
        tokenizer:
            The tokenizer used for training; will be saved and pushed along with
            the model.
        repo_id:
            Hugging Face Hub repository ID, e.g. "user/gemma-3-1b-gsm8k-grpo".
        output_dir:
            Local directory to save the merged model and tokenizer before upload.
        private:
            If True, create the Hub repo as private. If it already exists,
            its privacy setting is not changed.

    Returns:
        The merged `PreTrainedModel` instance, or None if merge failed.
    """
    # 1) Merge LoRA weights into the base model
    print("Merging LoRA adapters into base policy model...")
    try:
        merged_model: PreTrainedModel = trainer.model.merge_and_unload()
    except AttributeError as e:
        print("Error: trainer.model does not appear to be a PEFT model "
              "with a 'merge_and_unload' method.")
        print(f"Details: {e}")
        return None

    # 2) Save locally
    print(f"Saving merged model and tokenizer locally to: {output_dir}")
    merged_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    # 3) Ensure the Hub repository exists
    api = HfApi()
    create_repo(repo_id, private=private, exist_ok=True)

    # 4) Push model and tokenizer to the Hub
    print(f"Pushing merged policy model to Hugging Face Hub at: {repo_id}")
    merged_model.push_to_hub(repo_id)
    tokenizer.push_to_hub(repo_id)

    print("Upload complete. Policy model is now available on the Hub.")
    return merged_model

In [65]:
# =====================================================================
# Pre-training evaluation
# =====================================================================
print("🔍 Running pre-training evaluation on held-out test set...")
pre_metrics = evaluate_model_on_conll(
    model,
    tokenizer,
    eval_dataset,   
    prefix="pre_eval")
print("\n📊 Pre-training metrics:")
for k, v in pre_metrics.items():
    print(f"  {k}: {v:.4f}")

🔍 Running pre-training evaluation on held-out test set...

--------------------------------------------------------------------------------
evaluate_model_on_conll => prompt
[{'content': '\nYou are an information extraction system.\n\nYour task is to assign a BIO Named Entity tag to each input token.\n\nEntity types:\n- PER (Person)\n- ORG (Organization)\n- LOC (Location)\n- MISC (Miscellaneous)\n\nBIO rules:\n- Each entity must start with B-<TYPE>\n- Tokens inside the same entity must use I-<TYPE>\n- Tokens outside any entity must use O\n- Output exactly one label per input token\n- Do not add, remove, or reorder tokens\n\nAlways answer in this exact format:\n\n<reasoning>\nBriefly explain the main entity decisions you made.\nDo NOT explain every token.\n</reasoning>\n\n<answer>\nTOKEN<TAB>LABEL\n(one token per line, in the same order as input)\n</answer>\n\nRules:\n- Put ALL reasoning only between <reasoning> and </reasoning>.\n- The <answer> section must contain ONLY token–label pai

In [66]:
# Execute GRPO training with multi-reward optimization
print("🚀 Starting GRPO training...")
print("📊 Monitor metrics: reward scores, KL divergence, policy gradients")

checkpoint_path = "./trl_grpo_outputs/checkpoint-200"

# Run the training process
print(trainer.state.global_step)
trainer.train(resume_from_checkpoint=True)
print(trainer.state.global_step)

print("✅ Training completed successfully!")
print(f"💾 Model saved to: {training_args.output_dir}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting GRPO training...
📊 Monitor metrics: reward scores, KL divergence, policy gradients
0


Step,Training Loss
101,0.030600
102,0.019800
103,-0.000800
104,0.001500
105,0.006600
106,-0.006400
107,-0.002600
108,0.038600
109,-0.005600
110,0.000800


300
✅ Training completed successfully!
💾 Model saved to: ./trl_grpo_checkpoint_IL_blog_01


In [67]:
# =====================================================================
# Post-training evaluation
# =====================================================================
print("🔍 Running post-training evaluation on the same held-out test set...")
# Use trainer.model (same object as `model`, but updated by GRPO)
post_metrics = evaluate_model_on_conll(trainer.model, tokenizer, eval_dataset)

print("\n📊 Post-training metrics:")
for k, v in post_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\n📈 Summary (post - pre):")
for k in pre_metrics.keys():
    delta = post_metrics[k] - pre_metrics[k]
    print(f"  {k}: {pre_metrics[k]:.4f} → {post_metrics[k]:.4f} (Δ {delta:+.4f})")

🔍 Running post-training evaluation on the same held-out test set...

--------------------------------------------------------------------------------
evaluate_model_on_conll => prompt
[{'content': '\nYou are an information extraction system.\n\nYour task is to assign a BIO Named Entity tag to each input token.\n\nEntity types:\n- PER (Person)\n- ORG (Organization)\n- LOC (Location)\n- MISC (Miscellaneous)\n\nBIO rules:\n- Each entity must start with B-<TYPE>\n- Tokens inside the same entity must use I-<TYPE>\n- Tokens outside any entity must use O\n- Output exactly one label per input token\n- Do not add, remove, or reorder tokens\n\nAlways answer in this exact format:\n\n<reasoning>\nBriefly explain the main entity decisions you made.\nDo NOT explain every token.\n</reasoning>\n\n<answer>\nTOKEN<TAB>LABEL\n(one token per line, in the same order as input)\n</answer>\n\nRules:\n- Put ALL reasoning only between <reasoning> and </reasoning>.\n- The <answer> section must contain ONLY token

In [68]:
merged_model = save_policy_model_to_hf(
    trainer=trainer,
    tokenizer=tokenizer,
    repo_id=HF_REPO_ID,
    output_dir=OUTPUT_DIR,
    private=False,  # set True if you want a private repo
)

Merging LoRA adapters into base policy model...
Saving merged model and tokenizer locally to: ./trl_grpo_output_IL_blog_01
Pushing merged policy model to Hugging Face Hub at: Priyanlc


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Upload complete. Policy model is now available on the Hub.
